In [1]:
import os
import json
import pandas as pd



In [2]:
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI



In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [5]:
load_dotenv("../.env")

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [6]:
from src.rag import generate_rag_response

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set.")

client = OpenAI(
    api_key=OPENAI_API_KEY
)

JUDGE_MODEL = "gpt-5.4-nano"

In [8]:
with open(
    "../data/retrieval_ground_truth.json",
    "r"
) as f:

    ground_truth = json.load(f)

print(f"Loaded {len(ground_truth)} evaluation questions.")

Loaded 30 evaluation questions.


In [9]:
ground_truth[:3]

[{'question': 'A science fiction movie about space exploration',
  'relevant_movies': ['Interstellar', 'The Martian']},
 {'question': 'A mind-bending science fiction movie',
  'relevant_movies': ['Inception', 'Tenet']},
 {'question': 'A movie about time travel',
  'relevant_movies': ['Back to the Future', 'Looper']}]

In [10]:
test_question = ground_truth[0]["question"]

result = generate_rag_response(
    test_question
)

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED MOVIES:")

for movie in result["retrieved_movies"]:
    print("-", movie["title"])

print("\nANSWER:")
print(result["answer"])

QUESTION:
A science fiction movie about space exploration

RETRIEVED MOVIES:
- Interstellar: Nolan's Odyssey
- The Midnight Sky
- Passengers
- Ad Astra
- Orbiter 9

ANSWER:
### Best matches (space exploration / sci-fi)

**1) Ad Astra (2019-09-17)**  
- **Genre:** Science Fiction, Drama  
- **Director:** James Gray  
- **Rating:** 6.1  
- **Why it matches:** It follows an astronaut on a mission across space to uncover the truth about a lost expedition—directly aligning with **space exploration** themes.

**2) The Midnight Sky (2020-12-10)**  
- **Genre:** Science Fiction, Drama  
- **Director:** George Clooney  
- **Rating:** 5.766  
- **Why it matches:** Features a lone scientist trying to contact astronauts returning home amid a mysterious global catastrophe—strong **space/astronaut** focus.

**3) Passengers (2016-12-21)**  
- **Genre:** Drama, Romance, Science Fiction  
- **Director:** Morten Tyldum  
- **Rating:** 6.934  
- **Why it matches:** Set on a spacecraft traveling to a dist

In [11]:
#create llm judge prompt
def judge_rag_answer(
    question,
    context,
    answer
):

    judge_prompt = f"""
You are an evaluator for a movie recommendation RAG system.

Evaluate the generated answer using ONLY the provided
movie database context.

USER QUESTION:
{question}

RETRIEVED MOVIE CONTEXT:
{context}

GENERATED ANSWER:
{answer}

Evaluate the answer using these criteria.

1. Relevance
Score from 1 to 5.

5 = Directly and completely addresses the user's question.
4 = Mostly addresses the question with minor issues.
3 = Partially addresses the question.
2 = Mostly irrelevant.
1 = Completely irrelevant.

2. Correctness
Score from 1 to 5.

5 = All important claims are supported by the retrieved context.
4 = Mostly correct with minor issues.
3 = Some unsupported or questionable claims.
2 = Several incorrect claims.
1 = Mostly incorrect or fabricated.

3. Completeness
Score from 1 to 5.

5 = Provides all important information needed to answer the question.
4 = Mostly complete.
3 = Missing some useful information.
2 = Missing substantial information.
1 = Fails to provide the required information.

Return ONLY valid JSON in this format:

{{
    "relevance": 1,
    "correctness": 1,
    "completeness": 1,
    "explanation": "Brief explanation of the scores."
}}
"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,

        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict evaluator of RAG "
                    "system responses. Return valid JSON only."
                )
            },
            {
                "role": "user",
                "content": judge_prompt
            }
        ],

        temperature=0
    )

    content = response.choices[0].message.content

    return json.loads(content)

In [12]:
#test
evaluation = judge_rag_answer(
    result["question"],
    result["context"],
    result["answer"]
)

evaluation

{'relevance': 5,
 'correctness': 4,
 'completeness': 4,
 'explanation': "The answer is highly relevant, recommending multiple sci-fi/space exploration films from the provided context and explaining how each matches. Claims about plot/space elements for Ad Astra, The Midnight Sky, Passengers, and Orbiter 9 are supported by the retrieved overviews. However, it does not clearly recommend Interstellar: Nolan's Odyssey as a match (it only notes it is documentary), and the 'Best matches' framing is slightly inconsistent with the note about Interstellar being documentary rather than a fictional exploration story. Overall it covers the question well but could better incorporate the documentary option."}

In [13]:
#evaluate entire dataset
results = []

for item in tqdm(
    ground_truth,
    desc="Evaluating RAG"
):

    question = item["question"]

    try:

        rag_result = generate_rag_response(
            question
        )

        judge_result = judge_rag_answer(
            question,
            rag_result["context"],
            rag_result["answer"]
        )

        results.append({

            "question": question,

            "expected": ", ".join(
                item["relevant_movies"]
            ),

            "retrieved": ", ".join(
                movie["title"]
                for movie in rag_result["retrieved_movies"]
            ),

            "answer": rag_result["answer"],

            "relevance": judge_result["relevance"],

            "correctness": judge_result["correctness"],

            "completeness": judge_result["completeness"],

            "explanation": judge_result["explanation"]

        })

    except Exception as e:

        print(
            f"Error evaluating question: {question}"
        )

        print(e)

Evaluating RAG: 100%|██████████| 30/30 [02:07<00:00,  4.26s/it]


In [14]:
#create data frame
rag_results = pd.DataFrame(results)

rag_results.head()

,question,expected,retrieved,answer,relevance,correctness,completeness,explanation
0,A science fiction movie about space exploration,"Interstellar, The Martian","Interstellar: Nolan's Odyssey, The Midnight Sk...",### Best matches (science fiction + space expl...,5,4,4,The answer directly recommends multiple scienc...
1,A mind-bending science fiction movie,"Inception, Tenet","Bliss, Transcendence, Metropia, Coherence, Adv...",### Best match: **Coherence (2013)**\n- **Genr...,5,4,4,The answer directly recommends mind-bending sc...
2,A movie about time travel,"Back to the Future, Looper","Frequently Asked Questions About Time Travel, ...",## Best matches (time travel)\n\n### 1) **Abou...,5,4,4,The answer directly recommends multiple time-t...
3,A superhero movie with Marvel characters,"Avengers: Endgame, Iron Man","Marvel Studios: Assembling a Universe, Marvel ...",### Best match: **The Avengers (2012)**\n- **G...,5,5,4,The answer correctly identifies The Avengers a...
4,A superhero movie from DC,"The Dark Knight, Man of Steel",Lego Batman: The Movie - DC Super Heroes Unite...,### Best matches (DC superhero)\n\n1) **Zack S...,5,4,4,The answer directly recommends DC superhero mo...


In [15]:
#calculate final score
metrics = {

    "Average Relevance":
        rag_results["relevance"].mean(),

    "Average Correctness":
        rag_results["correctness"].mean(),

    "Average Completeness":
        rag_results["completeness"].mean()

}

metrics

{'Average Relevance': 4.933333333333334,
 'Average Correctness': 4.233333333333333,
 'Average Completeness': 3.933333333333333}

In [16]:
#overall score
overall_score = (
    metrics["Average Relevance"]
    + metrics["Average Correctness"]
    + metrics["Average Completeness"]
) / 3

print(
    f"Overall RAG Score: "
    f"{overall_score:.2f} / 5"
)

Overall RAG Score: 4.37 / 5


In [17]:
#display final evaluation
print("=" * 50)
print("LLM-AS-A-JUDGE EVALUATION")
print("=" * 50)

print(
    f"Average Relevance:    "
    f"{metrics['Average Relevance']:.2f} / 5"
)

print(
    f"Average Correctness:  "
    f"{metrics['Average Correctness']:.2f} / 5"
)

print(
    f"Average Completeness: "
    f"{metrics['Average Completeness']:.2f} / 5"
)

print(
    f"Overall Score:        "
    f"{overall_score:.2f} / 5"
)

LLM-AS-A-JUDGE EVALUATION
Average Relevance:    4.93 / 5
Average Correctness:  4.23 / 5
Average Completeness: 3.93 / 5
Overall Score:        4.37 / 5


In [18]:
output_path = "../data/rag_evaluation_results.csv"

rag_results.to_csv(
    output_path,
    index=False
)

print(
    f"Saved evaluation results to: {output_path}"
)

Saved evaluation results to: ../data/rag_evaluation_results.csv


In [19]:
#inspect bad answer
low_quality = rag_results[
    (
        rag_results["relevance"] <= 2
    )
    |
    (
        rag_results["correctness"] <= 2
    )
    |
    (
        rag_results["completeness"] <= 2
    )
]

print(
    f"Low-quality answers: "
    f"{len(low_quality)}"
)

low_quality[
    [
        "question",
        "retrieved",
        "answer",
        "relevance",
        "correctness",
        "completeness"
    ]
]

Low-quality answers: 0


,question,retrieved,answer,relevance,correctness,completeness


In [20]:
#best answer
high_quality = rag_results[
    (
        rag_results["relevance"] >= 4
    )
    &
    (
        rag_results["correctness"] >= 4
    )
    &
    (
        rag_results["completeness"] >= 4
    )
]

print(
    f"High-quality answers: "
    f"{len(high_quality)}"
)

High-quality answers: 27
